In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(palette="muted")

## Get The Dataset

In [ ]:
import yfinance as yf
from datetime import datetime

now = datetime.now()
start = datetime(now.year - 10, now.month, now.day)
end = now

ticker = "GOOG"
df = yf.download(ticker, start, end)
df.head()

### Data Cleansing

In [ ]:
df.isna().sum()

In [ ]:
df.dropna(inplace=True)
df.columns

In [ ]:
df.columns = df.columns.droplevel(1)
df.columns.names = [None]
df.head()

## Feature Engineering & Visualization

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(df, x=df.index, y="Close")
plt.title("GOOG")
plt.xlabel("Days")

In [ ]:
df["MA_100"] = df.Close.rolling(100).mean()
df["MA_200"] = df.Close.rolling(200).mean()
df.head(102)

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(df, x=df.index, y="MA_100", label="100 days")
sns.lineplot(df, x=df.index, y="MA_200", label="200 days")
plt.xlabel("Days")
plt.ylabel("Moving Average")

In [ ]:
df["pct_change"] = df.Close.pct_change()
df.head()

In [ ]:
plt.figure(figsize=(14, 5))
sns.lineplot(df, x=df.index, y="pct_change")
plt.xlabel("Days")
plt.ylabel("Percentage Change")

## Data Preproccessing

In [ ]:
df.shape

In [ ]:
df.Close

In [ ]:
from math import floor
# 30% test size
data = pd.DataFrame(df.Close)
p = 0.3
index = floor(len(data) * (1 - p))

train_data = data.iloc[:index]
test_data = data.iloc[index:]

print(train_data.shape, test_data.shape)

### Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler(feature_range=(0, 1))
train_data = scaler.fit_transform(train_data)

### Data Splitting

In [ ]:
x_train, y_train = [], []
for i in range(100, train_data.shape[0]):
    x_train.append(train_data[i-100 : i])
    y_train.append(train_data[i, 0])

x_train, y_train = np.array(x_train), np.array(y_train)

In [ ]:
x_train.shape

## Modeling

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, LSTM, Input

model = Sequential()

model.add(Input(shape=(100, 1)))
model.add(LSTM(128, activation="tanh", return_sequences=True))
model.add(LSTM(64))
model.add(Dense(25))
model.add(Dense(1))

model.compile(optimizer="adam", loss="mse")

In [ ]:
model.fit(x_train, y_train, epochs=50)
model.save("./stock-perdictor.keras")

In [ ]:
from keras.models import load_model
model = load_model("./stock-perdictor.keras")

In [ ]:
model.summary()

## Prediction

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))
test_data = scaler.fit_transform(test_data)

In [ ]:
test_data[:5]

In [ ]:
test_data = np.concat([train_data[-100:], test_data])
test_data.shape

In [ ]:
x_test, y_test = [], []
for i in range(100, test_data.shape[0]):
    x_test.append(test_data[i-100 : i])
    y_test.append(test_data[i, 0])

x_test, y_test = np.array(x_test), np.array(y_test)

In [ ]:
y_pred = model.predict(x_test)

## Evaluation

In [ ]:
y_test = scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_pred = scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()

In [ ]:
y_test[:5]

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(y_test, label="Real Price")
plt.plot(y_pred, label="Predicted Price")
plt.title("Real vs Prediction Price")
plt.xlabel("Days")
plt.ylabel("Price")
plt.legend()

Zoom in the plot

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(y_test, label="Real Price")
plt.plot(y_pred, label="Predicted Price")
plt.title("Real vs Prediction Price")
plt.xlabel("Days")
plt.ylabel("Price")
plt.xlim(550, 650)
plt.ylim(250, 400)
plt.legend()

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"{mse = }")
print(f"{rmse = }")
print(f"{r2 = }")